# Vault Secrets Sync to AWS using WIF and Doormat

Based on `hashicorp-services/vault-plugin-wif-examples/secret-sync/aws`. Doormat supplies temporary credentials only to Terraform so it can create AWS IAM resources. Vault stores no AWS access keys and exchanges its own signed identity token through `AssumeRoleWithWebIdentity`.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("./.env")

VAULT_TOKEN = os.getenv("VAULT_TOKEN")
VAULT_ADDR = os.getenv("VAULT_ADDR")
VAULT_CACERT = os.getenv("VAULT_CACERT")
AWS_REGION = os.getenv("AWS_REGION", "eu-central-1")
DOORMAT_AWS_ACCOUNT = "aws_jose.merchan_test"

os.environ["TF_VAR_aws_region"] = AWS_REGION
print(f"Doormat account alias: {DOORMAT_AWS_ACCOUNT}")
print(f"AWS region: {AWS_REGION}")

Doormat account alias: aws_jose.merchan_test
AWS region: eu-central-1


In [2]:
! vault status

Key                      Value
---                      -----
Seal Type                awskms
Recovery Seal Type       shamir
Initialized              true
Sealed                   false
Total Recovery Shares    1
Threshold                1
Version                  2.0.3+ent
Build Date               2026-06-16T21:32:56Z
Storage Type             raft
Cluster Name             vault-cluster-e89137a9
Cluster ID               b24418e0-d91d-d6d3-c571-5af49b1ec963
Removed From Cluster     false
HA Enabled               true
HA Cluster               https://vault-0.vault-internal:8201
HA Mode                  active
Active Since             2026-07-23T13:27:14.354277171Z
Raft Committed Index     18151
Raft Applied Index       18151
Last WAL                 7020


In [3]:
import json
import os
import shlex
import subprocess

subprocess.run(["doormat", "login", "-f"], check=True)
account = shlex.quote(DOORMAT_AWS_ACCOUNT)
result = subprocess.run(
    ["bash", "-lc", f'eval "$(doormat aws -a {account} export)" && env -0'],
    check=True,
    capture_output=True,
)
for entry in result.stdout.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

# Prevent the static-account notebook variables from entering the WIF destination.
os.environ.pop("TF_VAR_aws_access_key_id", None)
os.environ.pop("TF_VAR_aws_secret_access_key", None)

caller = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--output", "json"],
    check=True,
    capture_output=True,
    text=True,
)
identity = json.loads(caller.stdout)
print(json.dumps({"Account": identity["Account"], "Arn": identity["Arn"]}, indent=2))

time="2026-07-23T16:56:39+02:00" level=info msg="logging into doormat..."
time="2026-07-23T16:56:43+02:00" level=info msg="successfully logged into doormat!"


{
  "Account": "492487827579",
  "Arn": "arn:aws:sts::492487827579:assumed-role/aws_jose.merchan_test-developer/jose.merchan@hashicorp.com"
}


In [4]:
! terraform -chdir=terraform-wif init
! terraform -chdir=terraform-wif validate

Initializing the backend...

Initializing provider plugins...
- Reusing previous version of hashicorp/vault from the dependency lock file
- Reusing previous version of hashicorp/aws from the dependency lock file
- Reusing previous version of hashicorp/time from the dependency lock file
- Reusing previous version of hashicorp/tls from the dependency lock file
- Using previously-installed hashicorp/tls v4.3.0
- Using previously-installed hashicorp/vault v5.10.1
- Using previously-installed hashicorp/aws v6.56.0
- Using previously-installed hashicorp/time v0.14.0


Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necess

In [5]:
! terraform -chdir=terraform-wif plan

data.tls_certificate.issuer: Reading...
data.vault_namespace.current: Reading...
data.tls_certificate.issuer: Read complete after 0s [id=944d7a7937d502b17f4ba6bdafb60708b1917643]
data.aws_caller_identity.current: Reading...
data.aws_caller_identity.current: Read complete after 0s [id=492487827579]
data.vault_namespace.current: Read complete after 0s [id=/]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # aws_iam_openid_connect_provider.vault_secrets_sync will be created
  + resource "aws_iam_openid_connect_provider" "vault_secrets_sync" {
      + arn             = (known after apply)
      + client_id_list  = [
          + "vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync",
        ]
      + id              = (known after apply)
      + tags            = {
          + "managed-by" = "terraform-vault-secrets-sync-wif"

In [6]:
! terraform -chdir=terraform-wif apply -auto-approve

data.tls_certificate.issuer: Reading...
data.vault_namespace.current: Reading...
data.tls_certificate.issuer: Read complete after 0s [id=944d7a7937d502b17f4ba6bdafb60708b1917643]
data.aws_caller_identity.current: Reading...
data.aws_caller_identity.current: Read complete after 0s [id=492487827579]
data.vault_namespace.current: Read complete after 0s [id=/]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # aws_iam_openid_connect_provider.vault_secrets_sync will be created
  + resource "aws_iam_openid_connect_provider" "vault_secrets_sync" {
      + arn             = (known after apply)
      + client_id_list  = [
          + "vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync",
        ]
      + id              = (known after apply)
      + tags            = {
          + "managed-by" = "terraform-vault-secrets-sync-wif"

In [8]:
! vault read \
  sys/sync/destinations/aws-sm/mapfre-wif-aws-sm/associations

! aws secretsmanager get-secret-value \
  --secret-id "$(terraform -chdir=terraform-wif output -raw expected_aws_secret_name)" \
  --query SecretString \
  --output text

No value found at sys/sync/destinations/aws-sm/mapfre-wif-aws-sm/associations

aws: [ERROR]: An error occurred (ResourceNotFoundException) when calling the GetSecretValue operation: Secrets Manager can't find the specified secret.


In [10]:
%%bash
# Optional cleanup. Convert this cell to code only when you intend to destroy the WIF demo.
terraform -chdir=terraform-wif destroy -auto-approve

data.tls_certificate.issuer: Reading...
data.vault_namespace.current: Reading...
vault_activation_flags.secrets_sync: Refreshing state... [id=secrets-sync]
vault_mount.kv: Refreshing state... [id=mapfre-wif-kv]
vault_identity_oidc.issuer: Refreshing state... [id=https://vault.jose-merchan.sbx.hashidemos.io]
data.tls_certificate.issuer: Read complete after 0s [id=944d7a7937d502b17f4ba6bdafb60708b1917643]
data.aws_caller_identity.current: Reading...
data.aws_caller_identity.current: Read complete after 0s [id=492487827579]
data.vault_namespace.current: Read complete after 0s [id=/]
vault_kv_secret_v2.demo: Refreshing state... [id=mapfre-wif-kv/data/test]
vault_identity_oidc_key.secrets_sync: Refreshing state... [id=mapfre-wif-secrets-sync-key]
aws_iam_openid_connect_provider.vault_secrets_sync: Refreshing state... [id=arn:aws:iam::492487827579:oidc-provider/vault.jose-merchan.sbx.hashidemos.io/v1/identity/oidc/secrets-sync]
vault_identity_oidc_role.publish_key: Refreshing state... [id=ma